# 💰 Python Greedy Algorithms — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Greedy is the "always take the best deal on the table right now" strategy. Imagine running a food truck: at each intersection you drive toward the nearest hungry crowd instead of mapping every possible route. It works when local best choices chain together into a global best — and the key skill is *proving* that greedily committing never forecloses a better option. When it works, you get O(n log n). When it doesn't, you need DP.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Greedy? The Visual Model](#1) |
| 2 | [Creating / Setup — Greedy Scaffolds](#2) |
| 3 | [The Core API — Sort, Scan, Commit](#3) |
| 4 | [Decision Map — When Greedy Works](#4) |
| 5 | [Pattern 1: Jump Game (LC 55)](#5) |
| 6 | [Pattern 2: Jump Game II — Min Jumps (LC 45)](#6) |
| 7 | [Pattern 3: Gas Station (LC 134)](#7) |
| 8 | [Pattern 4: Non-overlapping Intervals (LC 435)](#8) |
| 9 | [Pattern 5: Partition Labels (LC 763)](#9) |
| 10 | [The Greedy Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Greedy? The Visual Model

```
JUMP GAME GREEDY — nums = [2, 3, 1, 1, 4]

  idx:    0    1    2    3    4
  jump:  [2]  [3]  [1]  [1]  [4]
         │    │    │    │    │
  reach: 2    4    3    4    8

  max_reach tracks the farthest index reachable so far:
  i=0: max_reach = max(0, 0+2)=2
  i=1: max_reach = max(2, 1+3)=4   ← best jump from this position
  i=2: max_reach = max(4, 2+1)=4   (no improvement)
  i=3: max_reach = max(4, 3+1)=4   (no improvement)
  i=4: 4 >= n-1=4 → REACHABLE ✓

INTERVAL SCHEDULING — sort by END time, greedily keep non-overlapping:

  Sorted by end:  [(1,2),(1,3),(2,3),(3,4)]
  keep (1,2):  last_end=2
  skip (1,3):  starts at 1 < last_end=2  → overlap
  keep (2,3):  starts at 2 = last_end=2  → no overlap, last_end=3
  keep (3,4):  starts at 3 = last_end=3  → no overlap, last_end=4
  kept=3, removed=4-3=1

  WHY END TIME? An interval that ends earliest leaves the most room for future
  intervals. Sorting by start or length gives suboptimal results.

GREEDY CHOICE PROPERTY:
  Greedy works when: "there always exists an optimal solution that includes
  the greedy choice." Prove by exchange argument: if optimal solution
  differs, swapping to the greedy choice never makes it worse.
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Greedy Scaffolds

In [ ]:
# GREEDY SCAFFOLDS — three common shapes

# Shape 1: Single-pass max_reach tracker (jump game family)
def greedy_reach(nums):
    max_reach = 0              # farthest index reachable so far
    for i, jump in enumerate(nums):
        if i > max_reach:
            return False       # current position is unreachable — stuck
        max_reach = max(max_reach, i + jump)
    return True

# Shape 2: Sort then scan (interval scheduling family)
def greedy_intervals(intervals):
    intervals.sort(key=lambda x: x[1])   # sort by END time — key insight
    last_end = float('-inf')
    kept = 0
    for start, end in intervals:
        if start >= last_end:             # no overlap with last kept
            kept += 1
            last_end = end
    return kept

# Shape 3: Running sum with reset (gas station family)
def greedy_circular(gains):
    total = 0           # global sum — if >= 0, solution exists
    running = 0         # running sum from current candidate start
    start = 0           # candidate starting position
    for i, g in enumerate(gains):
        total += g
        running += g
        if running < 0:         # can't reach here from current start
            start = i + 1       # try starting from next position
            running = 0         # reset running sum
    return start if total >= 0 else -1

print("Shape 1 - reach: can_reach([2,3,1,1,4])=", greedy_reach([2,3,1,1,4]))   # True
print("Shape 1 - reach: can_reach([3,2,1,0,4])=", greedy_reach([3,2,1,0,4]))   # False
print("Shape 2 - intervals: kept=", greedy_intervals([(1,2),(1,3),(2,3),(3,4)]))  # 3
print("Shape 3 - circular: start=", greedy_circular([-2,1,-3,4,-1]))            # 3
print("Greedy scaffolds defined.")

<a id='3'></a>
## 3. ⚡ The Core API — Sort, Scan, Commit

```
OPERATION                         COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────────
intervals.sort(key=lambda x: x[1]) O(n log n)  sort intervals by END — key for scheduling
max_reach = max(reach, i + nums[i]) O(1)        track farthest reachable index
if i > max_reach: return False      O(1)        detect unreachable position (stuck)
total >= 0 → solution exists        O(1)        Bezout-like global feasibility check
if running < 0: start = i+1         O(1)        reset candidate start when stuck
last[c] = last occurrence index     O(n)        precompute for partition problems
──────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  sort intervals by START for scheduling — gives wrong results; sort by END
✅  sort by END: keeps max future options open (exchange argument proof)
❌  use DP when greedy suffices — greedy is O(n log n), DP is O(n²)
✅  check greedy choice property first: can local optimal = global optimal?
❌  float comparisons for interval endpoints — use integers or exact fractions
❌  forget to handle the edge case where all elements are 0 in jump game
✅  check i > max_reach inside the loop, not just at the end
```

In [ ]:
# LIVE DEMO: why sorting by END (not START or LENGTH) is optimal for intervals

intervals = [(1, 4), (2, 3), (3, 5), (1, 2)]

def count_kept(intervals, sort_key):
    s = sorted(intervals, key=sort_key)
    last_end = float('-inf')
    kept = 0
    for start, end in s:
        if start >= last_end:
            kept += 1
            last_end = end
    return kept, s

k_end, s_end = count_kept(intervals, lambda x: x[1])    # sort by end
k_start, s_start = count_kept(intervals, lambda x: x[0]) # sort by start
k_len, s_len = count_kept(intervals, lambda x: x[1]-x[0]) # sort by length

print(f"Sort by END:    {s_end} → kept={k_end}   ← CORRECT")
print(f"Sort by START:  {s_start} → kept={k_start}")
print(f"Sort by LENGTH: {s_len} → kept={k_len}")
print()
print("Exchange argument: if optimal solution skips the earliest-ending interval,")
print("we can swap in the earliest-ending one without reducing the count — it fits")
print("in the same slot and ends earlier, never blocking more future intervals.")

<a id='4'></a>
## 4. 🗂️ Decision Map — When Greedy Works

```
PROBLEM SIGNAL                            GREEDY APPROACH
──────────────────────────────────────────────────────────────────────
"can you reach the end" (jump game)       max_reach single scan
"minimum jumps to reach end"              BFS-style: boundary + jumps counter
"find valid start in circular problem"    running sum + reset + total check
"remove min intervals to avoid overlap"  sort by end, count kept
"partition string by last occurrence"     last[] map + extend window
"assign items to maximize satisfaction"  sort both, two-pointer match
──────────────────────────────────────────────────────────────────────

GREEDY vs DP DECISION:
  Greedy works:  problem has "greedy choice property" + "optimal substructure"
  DP needed:     choices depend on future state (e.g., 0/1 knapsack)

  Ask: "If I make the locally optimal choice now, does it ever close off
        a better global solution?" If NO → greedy. If YES → DP.

  Jump game: taking max reach now never hurts → greedy ✓
  0/1 Knapsack: taking heaviest item may block better combo → DP ✗

SORTING KEYS:
  Interval scheduling → sort by END time
  Meeting rooms (can attend max) → sort by START, use min-heap for end times
  Task scheduling by deadline → sort by DEADLINE (EDF algorithm)
  Huffman encoding → sort by FREQUENCY (min-heap)
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Jump Game — LC 55

---

```
PROBLEM:  Given nums where nums[i] = max jump length from index i,
          can you reach the last index starting from index 0?
APPROACH: Track max_reach = farthest index reachable from any position seen so far.
          If current index i > max_reach, position is unreachable → False.
          Update max_reach = max(max_reach, i + nums[i]) at each step.

SLOW MOTION TRACE on nums = [3, 2, 1, 0, 4]:

  i=0: nums[0]=3, max_reach = max(0, 0+3)=3.  i=0 ≤ 3 → ok
  i=1: nums[1]=2, max_reach = max(3, 1+2)=3.  i=1 ≤ 3 → ok
  i=2: nums[2]=1, max_reach = max(3, 2+1)=3.  i=2 ≤ 3 → ok
  i=3: nums[3]=0, max_reach = max(3, 3+0)=3.  i=3 ≤ 3 → ok
  i=4: i=4 > max_reach=3 → UNREACHABLE → return False ✓

  nums[3]=0 creates a wall at index 3 — can reach index 3 but no farther.

SLOW MOTION TRACE on nums = [2, 3, 1, 1, 4]:
  i=0: max_reach=2
  i=1: max_reach=max(2,4)=4
  i=2: max_reach=max(4,3)=4
  i=3: max_reach=max(4,4)=4
  End of loop, 4 >= n-1=4 → True ✓

KEY INSIGHT: max_reach is a moving horizon. The instant i surpasses it,
             no jump from any earlier position can bridge the gap.
TIME:  O(n) — single pass
SPACE: O(1) — just max_reach variable
```

In [ ]:
def can_jump(nums):
    """
    LC 55 — Jump Game
    Approach: greedy max_reach scan — track farthest reachable index.
    Args:
        nums (List[int]): nums[i] = max jump from index i.
    Returns:
        bool: True if last index is reachable from index 0.
    Time:  O(n) — single pass
    Space: O(1) — one variable
    """
    max_reach = 0   # farthest index we can currently reach

    for i, jump in enumerate(nums):
        if i > max_reach:
            return False            # i is beyond our horizon — wall hit
        max_reach = max(max_reach, i + jump)  # extend horizon if this jump is better

    return True   # survived the entire loop — last index reachable

# Slow motion on nums=[3,2,1,0,4]:
# i=0: max_reach=3  i=1: max_reach=3  i=2: max_reach=3
# i=3: max_reach=3  i=4: 4>3 → False  (nums[3]=0 creates the wall)

# Slow motion on nums=[2,3,1,1,4]:
# i=0: max_reach=2  i=1: max_reach=4  i=2,3: no improvement
# loop ends normally → True

def test_harness(fn):
    tests = [
        ([2, 3, 1, 1, 4], True),
        ([3, 2, 1, 0, 4], False),
        ([0], True),               # single element — already at end
        ([1, 0], True),            # jump 1 from index 0 → reach index 1
        ([0, 1], False),           # stuck at index 0 (nums[0]=0)
        ([1, 1, 1, 0], True),      # just barely reach end
        ([2, 0, 0], True),         # jump over the zeros
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(can_jump)
print("can_jump defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Jump Game II — Minimum Jumps — LC 45

---

```
PROBLEM:  Given nums where nums[i] = max jump from i, return the minimum
          number of jumps to reach the last index. Guaranteed reachable.
APPROACH: BFS-style greedy. Each "level" = one jump. Track:
          - current_end: farthest index reachable in current level
          - farthest: farthest index reachable in NEXT level
          When i reaches current_end, take one more jump (jump to farthest).

SLOW MOTION TRACE on nums = [2, 3, 1, 1, 4]:

  jumps=0, current_end=0, farthest=0

  i=0: farthest=max(0,0+2)=2.  i=0==current_end=0 → take jump → jumps=1, current_end=2
  i=1: farthest=max(2,1+3)=4.  i=1 < current_end=2
  i=2: farthest=max(4,2+1)=4.  i=2==current_end=2 → take jump → jumps=2, current_end=4
  i=3: farthest=max(4,3+1)=4.  i=3 < current_end=4
  (stop before last index)

  Result: 2 jumps ✓  (0→1→4 or 0→1→3→4, but greedy picks min)

  LEVEL VIEW (like BFS levels):
    Level 0: [0]          (start)
    Level 1: [1, 2]       (reachable in 1 jump from 0: indices 1,2)
    Level 2: [3, 4, 5]    (reachable in 2 jumps: from 1→4, from 2→3)
    Index 4 appears at level 2 → answer = 2

KEY INSIGHT: current_end is the BFS frontier. We only increment jumps when
             we exhaust the current frontier. farthest is already the next frontier.
TIME:  O(n) — single pass
SPACE: O(1)
```

In [ ]:
def jump(nums):
    """
    LC 45 — Jump Game II
    Approach: greedy BFS levels — increment jumps when exhausting current frontier.
    Args:
        nums (List[int]): nums[i] = max jump from index i. Guaranteed reachable.
    Returns:
        int: minimum number of jumps to reach last index.
    Time:  O(n) — single pass
    Space: O(1) — three integer variables
    """
    jumps = 0
    current_end = 0   # farthest index reachable with `jumps` jumps (current level end)
    farthest = 0      # farthest index reachable with `jumps+1` jumps (next level end)

    for i in range(len(nums) - 1):   # stop before last index — no need to jump from there
        farthest = max(farthest, i + nums[i])   # extend next level's reach

        if i == current_end:          # exhausted current level — must jump
            jumps += 1
            current_end = farthest    # advance frontier to next level

    return jumps

# Slow motion on nums=[2,3,1,1,4]:
# i=0: farthest=2, i==current_end=0 → jumps=1, current_end=2
# i=1: farthest=4
# i=2: farthest=4, i==current_end=2 → jumps=2, current_end=4
# i=3: farthest=4
# loop ends (stop before index 4=n-1)
# return 2 ✓

# Slow motion on nums=[2,3,0,1,4]:
# i=0: farthest=2, jump → jumps=1, current_end=2
# i=1: farthest=4
# i=2: farthest=max(4,2+0)=4, i==current_end=2 → jumps=2, current_end=4
# i=3: farthest=max(4,4)=4
# return 2 ✓ (0→1→4)

def test_harness(fn):
    tests = [
        ([2, 3, 1, 1, 4], 2),
        ([2, 3, 0, 1, 4], 2),
        ([0], 0),                # single element — already there
        ([1, 2], 1),             # one jump
        ([1, 1, 1, 1], 3),       # must take every step
        ([4, 0, 0, 0, 1], 1),    # single long jump
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(jump)
print("jump defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Gas Station — LC 134

---

```
PROBLEM:  N gas stations in a circle. Each station i has gas[i] liters.
          Travel from i to i+1 costs cost[i]. Find starting station for
          a complete clockwise circuit, or return -1 if impossible.
APPROACH: Two key insights:
          1. If sum(gas - cost) >= 0, exactly one valid start exists.
          2. Find it: if running sum goes negative, reset start to i+1.

SLOW MOTION TRACE on gas=[1,2,3,4,5], cost=[3,4,5,1,2]:

  net gains = gas - cost = [-2,-2,-2,3,3]
  total = -2-2-2+3+3 = 0 ≥ 0 → solution exists

  scan for start:
  i=0: running=-2 < 0 → reset start=1, running=0
  i=1: running=-2 < 0 → reset start=2, running=0
  i=2: running=-2 < 0 → reset start=3, running=0
  i=3: running=3  ≥ 0 → ok
  i=4: running=6  ≥ 0 → ok
  result: start=3 ✓

  Verify: start at 3, gas=4, cost=1 → tank=3
    →4: tank=3+5-2=6  →0: tank=6+1-3=4  →1: tank=4+2-4=2  →2: tank=2+3-5=0 ✓

WHY THIS WORKS:
  If we can't complete circuit starting at A (running sum goes negative
  at station B), then no station between A and B can be the start either.
  Proof: any station C between A and B has positive prefix sum from A
  (else we'd have reset before reaching C). So starting at C gives
  LESS net gain at B than starting at A — still can't pass B.
  Thus, skip all of [A..B] and try B+1.

TIME:  O(n) — single pass
SPACE: O(1)
```

In [ ]:
def can_complete_circuit(gas, cost):
    """
    LC 134 — Gas Station
    Approach: if total net gain >= 0, exactly one start exists;
              find it by resetting candidate start whenever running sum < 0.
    Args:
        gas (List[int]): gas available at each station.
        cost (List[int]): gas cost to travel to next station.
    Returns:
        int: valid starting station index, or -1 if impossible.
    Time:  O(n) — single pass
    Space: O(1)
    """
    total = 0     # global feasibility: if >= 0, answer exists
    running = 0   # net fuel from current candidate start
    start = 0     # current candidate starting station

    for i in range(len(gas)):
        net = gas[i] - cost[i]   # net gain at station i
        total += net
        running += net

        if running < 0:
            # can't reach station i+1 from current start
            # skip entire [start..i] — none of them can be valid start
            start = i + 1
            running = 0   # fresh slate from new candidate

    return start if total >= 0 else -1

# Slow motion on gas=[1,2,3,4,5], cost=[3,4,5,1,2]:
# net=[-2,-2,-2,3,3]  total=0 after loop
# i=0: running=-2→reset start=1  i=1: running=-2→reset start=2
# i=2: running=-2→reset start=3  i=3: running=3  i=4: running=6
# total=0>=0 → return start=3 ✓

def test_harness(fn):
    tests = [
        ([1, 2, 3, 4, 5], [3, 4, 5, 1, 2], 3),
        ([2, 3, 4], [3, 4, 3], -1),              # total=-1 < 0 → impossible
        ([5, 1, 2, 3, 4], [4, 4, 1, 5, 1], 4),
        ([1, 2], [2, 1], 1),                     # start at station 1
        ([3, 1, 1], [1, 2, 2], 0),               # start at 0
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | gas={inputs[0]},cost={inputs[1]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(can_complete_circuit)
print("can_complete_circuit defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Non-overlapping Intervals — LC 435

---

```
PROBLEM:  Given intervals, find the minimum number of intervals to REMOVE
          so the remaining intervals are non-overlapping.
APPROACH: Equivalent to: maximize intervals KEPT (non-overlapping).
          Sort by END time. Greedily keep each interval that doesn't overlap
          with the last kept one. Answer = total - kept.

SLOW MOTION TRACE on intervals=[[1,2],[2,3],[3,4],[1,3]]:

  sorted by end: [[1,2],[2,3],[1,3],[3,4]]

  last_end = -inf, kept=0
  [1,2]: start=1 >= -inf → keep, last_end=2, kept=1
  [2,3]: start=2 >= last_end=2 → keep, last_end=3, kept=2
  [1,3]: start=1 < last_end=3 → SKIP (overlaps, and ends later — greedy rejects)
  [3,4]: start=3 >= last_end=3 → keep, last_end=4, kept=3

  kept=3, total=4 → remove = 4-3 = 1 ✓

TRACE on intervals=[[1,2],[1,2],[1,2]]:
  sorted by end: same
  [1,2]: keep, last_end=2
  [1,2]: 1 < 2 → skip
  [1,2]: 1 < 2 → skip
  kept=1 → remove=2 ✓

KEY INSIGHT: An interval that ends earlier ALWAYS gives at least as much
             room for future intervals. This is the greedy choice property.
             (Exchange argument: swap any kept interval for an earlier-ending one
             that's compatible — can only improve or stay equal.)
TIME:  O(n log n) — dominated by sort
SPACE: O(1) or O(log n) for sort
```

In [ ]:
def erase_overlap_intervals(intervals):
    """
    LC 435 — Non-overlapping Intervals
    Approach: sort by end, greedily keep non-overlapping; answer = total - kept.
    Args:
        intervals (List[List[int]]): list of [start, end] intervals.
    Returns:
        int: minimum number of intervals to remove.
    Time:  O(n log n) — sorting dominates
    Space: O(1)       — just last_end and counter
    """
    if not intervals:
        return 0

    intervals.sort(key=lambda x: x[1])   # sort by END — greedy choice key

    kept = 0
    last_end = float('-inf')   # end time of last kept interval

    for start, end in intervals:
        if start >= last_end:             # no overlap with last kept interval
            kept += 1
            last_end = end               # this becomes the new "last kept" boundary
        # else: overlap → skip this interval (it ends later than last_end, worse choice)

    return len(intervals) - kept   # minimum removals = total - max kept

# Slow motion on [[1,2],[2,3],[3,4],[1,3]]:
# sorted by end: [[1,2],[2,3],[1,3],[3,4]]
# [1,2]: 1>=-inf → keep, last_end=2
# [2,3]: 2>=2 → keep, last_end=3
# [1,3]: 1<3 → skip
# [3,4]: 3>=3 → keep, last_end=4
# kept=3 → return 4-3=1 ✓

def test_harness(fn):
    tests = [
        ([[1,2],[2,3],[3,4],[1,3]], 1),
        ([[1,2],[1,2],[1,2]], 2),
        ([[1,2],[2,3]], 0),                  # already non-overlapping
        ([[1,100],[11,22],[1,11],[2,12]], 2),
        ([], 0),                             # empty
        ([[1,2]], 0),                        # single interval
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(erase_overlap_intervals)
print("erase_overlap_intervals defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Partition Labels — LC 763

---

```
PROBLEM:  Partition string s into as many parts as possible so that each
          letter appears in at most one part. Return partition sizes.
APPROACH: Precompute last[c] = last index where character c appears.
          Scan left-to-right, maintaining a window [window_start..window_end].
          window_end = max(window_end, last[s[i]]) at each step.
          When i == window_end, the current partition is complete.

SLOW MOTION TRACE on s = "ababcbacadefegdehijhklij":

  last: a→8, b→5, c→7, d→14, e→15, f→11, g→13, h→19, i→22, j→23, k→20, l→21

  window_start=0, window_end=0
  i=0 'a': window_end=max(0,8)=8
  i=1 'b': window_end=max(8,5)=8
  i=2 'a': window_end=max(8,8)=8
  i=3 'b': window_end=max(8,5)=8
  i=4 'c': window_end=max(8,7)=8
  i=5 'b': window_end=max(8,5)=8
  i=6 'a': window_end=max(8,8)=8
  i=7 'c': window_end=max(8,7)=8
  i=8 'a': window_end=max(8,8)=8.  i==window_end → PARTITION!  size=8-0+1=9
           window_start=9
  i=9 'd': window_end=max(9,14)=14
  ...continues until i=14 for second partition (size 7)
  ...third partition size 8
  result = [9, 7, 8] ✓

KEY INSIGHT: window_end is pulled to the right every time we see a character
             that appears later in the string. When i catches up to window_end,
             all characters in [window_start..i] have their last occurrence inside
             the window — safe to cut here.
TIME:  O(n) — one pass for last[], one pass to partition
SPACE: O(1) — last[] has at most 26 entries (fixed alphabet)
```

In [ ]:
def partition_labels(s):
    """
    LC 763 — Partition Labels
    Approach: last occurrence map + greedy window expansion; cut when window closes.
    Args:
        s (str): lowercase English letters string.
    Returns:
        List[int]: sizes of each partition in order.
    Time:  O(n) — two passes (build last[], scan partitions)
    Space: O(1) — last[] has at most 26 entries
    """
    last = {c: i for i, c in enumerate(s)}   # last[c] = rightmost index of char c

    result = []
    window_start = 0
    window_end = 0

    for i, c in enumerate(s):
        window_end = max(window_end, last[c])   # pull window right to cover last c

        if i == window_end:                      # window closed — all chars accounted for
            result.append(window_end - window_start + 1)
            window_start = i + 1                 # start fresh after this partition

    return result

# Slow motion on s="abac":
# last: a→2, b→1, c→3
# i=0 'a': window_end=2
# i=1 'b': window_end=max(2,1)=2
# i=2 'a': window_end=max(2,2)=2.  i==2 → partition size=3, window_start=3
# i=3 'c': window_end=3.  i==3 → partition size=1
# result=[3,1]

# Slow motion on s="eccbbbbedffff":
# last: e→8,c→4,b→7,d→9,f→12
# window expands until i=9 (d's last occurrence is 9) → first partition size=10
# then f at 10..12 → second partition size=3

def test_harness(fn):
    tests = [
        ("ababcbacadefegdehijhklij", [9, 7, 8]),
        ("eccbbbbedffff", [10, 3]),
        ("a", [1]),
        ("ab", [1, 1]),                   # each letter appears once in its own part
        ("abac", [3, 1]),
        ("aaaa", [4]),                    # all same letter → one partition
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(partition_labels)
print("partition_labels defined.")

<a id='10'></a>
## 10. 🗺️ The Greedy Decision Map

```
QUESTION TYPE                    KEY TECHNIQUE              LC PROBLEMS
────────────────────────────────────────────────────────────────────────────
Can you reach the end?           max_reach scan             55 Jump Game
Min jumps to reach end           BFS boundary + jumps       45 Jump Game II
Valid start in circular route    running sum + reset        134 Gas Station
Min removals for non-overlap     sort by end, count kept    435 Non-overlapping
Max partitions by last occurrence last[] + window expand   763 Partition Labels
────────────────────────────────────────────────────────────────────────────

GREEDY PROOF TECHNIQUE — Exchange Argument:
  1. Assume optimal solution O exists that differs from greedy solution G.
  2. Find first point of difference: O chooses X, G chooses Y (greedily better).
  3. Show swapping X→Y in O produces O' that is at least as good as O.
  4. Repeat until O = G — greedy is optimal.

  Example (interval scheduling):
  O keeps interval X (ends later). G keeps Y (ends earlier, same start).
  Swap X→Y in O: Y ends earlier → at least as many future intervals fit.
  O' is valid and no worse → G is optimal. □

GREEDY FAILS WHEN:
  - Local choice forecloses globally better paths (0/1 knapsack)
  - Problem requires counting subsets/combinations (use DP)
  - Constraints couple non-adjacent elements (use DP or backtracking)
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for Greedy:

| Signal | What to Do |
|--------|------------|
| "can you reach end" | max_reach scan — O(n) |
| "min jumps to end" | BFS levels — current_end + farthest |
| "find start of circular route" | running sum reset + total check |
| "remove min intervals" | sort by end, count kept |
| "max partitions" | last occurrence + window |
| "assign to maximize happiness" | sort both, match greedily |

### The O(1) operations — memorize these:

```python
max_reach = max(max_reach, i + nums[i])  # jump game horizon
if i > max_reach: return False           # stuck — unreachable
if i == current_end: jumps += 1; current_end = farthest  # take a jump
if start >= last_end: kept += 1; last_end = end          # keep interval
if running < 0: start = i+1; running = 0  # gas station reset
last = {c: i for i, c in enumerate(s)}  # last occurrence map
```

### Common templates:

```python
# TEMPLATE 1: MAX REACH (jump game family)
max_reach = 0
for i, jump in enumerate(nums):
    if i > max_reach: return False  # stuck
    max_reach = max(max_reach, i + jump)

# TEMPLATE 2: BFS LEVELS (min jumps)
jumps, cur_end, farthest = 0, 0, 0
for i in range(len(nums)-1):
    farthest = max(farthest, i+nums[i])
    if i == cur_end: jumps += 1; cur_end = farthest

# TEMPLATE 3: INTERVAL SCHEDULING (max kept / min removed)
intervals.sort(key=lambda x: x[1])  # sort by END
last_end = float('-inf'); kept = 0
for start, end in intervals:
    if start >= last_end: kept += 1; last_end = end
removed = len(intervals) - kept

# TEMPLATE 4: CIRCULAR RUNNING SUM (gas station)
total = running = 0; start = 0
for i, net in enumerate(g - c for g, c in zip(gas, cost)):
    total += net; running += net
    if running < 0: start = i+1; running = 0
return start if total >= 0 else -1
```

### Gotchas to not forget:

```
❌  sort intervals by START for scheduling — wrong; always sort by END
✅  sort by END: leaves maximum room for future intervals
❌  jump game: check max_reach >= n-1 OUTSIDE the loop — check i>max_reach INSIDE
✅  check i > max_reach at each iteration — catches zero-jump walls
❌  jump game II: iterate all the way to len(nums)-1 — skip last index (no need to jump from there)
✅  for i in range(len(nums) - 1)
❌  gas station: check total INSIDE the loop — check AFTER the full scan
✅  total and running are separate: total for feasibility, running for start detection
❌  partition labels: use float for window_end — use integer index
✅  window_end = max(window_end, last[c]) at every step
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
                       GREEDY ALGORITHMS
                              │
           ┌──────────────────┼──────────────────┐
           │                  │                  │
        REACH              INTERVAL           CIRCULAR
       TRACKING            SCHEDULING         RUNNING SUM
           │                  │                  │
       max_reach          sort by END          running
       i > max_reach      keep if start        if<0: reset
       → False            >= last_end          total>=0?
           │                  │                  │
       LC 55             LC 435               LC 134
       LC 45 (BFS levels)

           WINDOW EXPANSION
           last[c] map
           extend window_end
           cut when i==window_end
           LC 763

GREEDY PROOF TEMPLATE:
  Assume optimal ≠ greedy.
  Find first difference.
  Exchange greedy choice in.
  Show result is no worse.
  Conclude greedy is optimal. □
```

---
*End of Greedy Algorithms Master Guide — Sean Edition*